# ODI to Databricks Migration: `SCEN_TASK_NO in {10}`

**Conversion Timestamp:** 2024-07-30 12:00:00

**Description:** This notebook performs a direct insert into the `TRG_LOC` table from the `LOCATIONS` table, mimicking a full load operation without intermediate staging/flow tables. Original ODI hints have been removed and Oracle syntax converted to Spark SQL.

In [ ]:
dbutils.widgets.text("ETL_JOB_TYPE", "", "1. ETL Job Type")
dbutils.widgets.text("DATASOURCE_NUM_ID", "0", "2. Datasource Number ID")
dbutils.widgets.text("ETL_PROC_WID", "0", "3. ETL Process WID")
dbutils.widgets.text("ODI_SESS_NO", "0", "4. ODI Session Number")

## ETL Parameters

In [ ]:
print(f"ETL_JOB_TYPE: {dbutils.widgets.get('ETL_JOB_TYPE')}")
print(f"DATASOURCE_NUM_ID: {dbutils.widgets.get('DATASOURCE_NUM_ID')}")
print(f"ETL_PROC_WID: {dbutils.widgets.get('ETL_PROC_WID')}")
print(f"ODI_SESS_NO: {dbutils.widgets.get('ODI_SESS_NO')}")

## Target Table DDL (`TRG_LOC`)

In [ ]:
%sql
DROP TABLE IF EXISTS workspace.hr.trg_loc;

In [ ]:
%sql
CREATE TABLE workspace.hr.trg_loc (
    LOCATION_ID     BIGINT,
    STREET_ADDRESS  STRING,
    POSTAL_CODE     STRING,
    CITY            STRING,
    STATE_PROVINCE  STRING,
    COUNTRY_ID      STRING
)
USING DELTA;

## Insert into Target (`TRG_LOC`)

In [ ]:
%sql
-- SCEN_TASK_NO in {10}
-- SCEN_TASK_NO in {20}
-- SCEN_TASK_NO in {30}
INSERT INTO workspace.hr.trg_loc
  (
    LOCATION_ID ,
    STREET_ADDRESS ,
    POSTAL_CODE ,
    CITY ,
    STATE_PROVINCE ,
    COUNTRY_ID
  )
SELECT
  locations.location_id ,
  locations.street_address ,
  locations.postal_code ,
  locations.city ,
  locations.state_province ,
  locations.country_id
FROM
  workspace.hr.locations AS locations;

## Validation

In [ ]:
%sql
SELECT COUNT(*) FROM workspace.hr.trg_loc;

## Cleanup

In [ ]:
%sql
-- No specific intermediate staging or flow tables were used in this direct insert.
-- Therefore, no cleanup is required beyond the target table itself, which was handled at the start.

## Conversion Notes and Manual Actions Required

1.  **Schema and Table Names:** All schema and table names have been converted to lowercase and prefixed with `workspace.`. (e.g., `HR.TRG_LOC` -> `workspace.hr.trg_loc`).
2.  **Oracle Hints Removed:** The `/*+ APPEND PARALLEL */` hint has been removed as it is specific to Oracle and not applicable to Databricks Delta Lake.
3.  **Data Type Mapping:** Assumed DDL for `TRG_LOC` based on typical `HR.LOCATIONS` column types. `NUMBER(p,0)` converted to `BIGINT`, `VARCHAR2` and `CHAR` to `STRING`. Please verify the actual source DDL for `HR.LOCATIONS` and update the `CREATE TABLE` statement for `workspace.hr.trg_loc` if necessary.
4.  **Full Load:** This script performs a full load by dropping and recreating the `trg_loc` table before inserting all data from `locations`.
5.  **SCEN_TASK_NO:** The `SCEN_TASK_NO in {10}`, `{20}`, `{30}` markers from the original ODI file, which did not contain any SQL, are included as comments in the main `INSERT` SQL cell for context.